![Imgur](https://i.imgur.com/acSOZRh.png)

# Laboratorio n° 4. Parte B: Segmentación semántica con U-Net

**Asignatura:** Redes Neuronales Profundas
**Bloque:** 4 — Detección y segmentación

---

## Introducción

En el laboratorio anterior trabajamos con **detección de objetos**: el modelo predice una caja (bounding box) por cada objeto de la imagen. En este vamos un paso más allá: **segmentación semántica**, donde el modelo asigna una clase a **cada píxel** de la imagen. En lugar de "hay un perro en esta caja" la predicción pasa a ser "estos píxeles son perro, estos son fondo, estos son sofá".

La diferencia es importante: una caja siempre incluye píxeles que no pertenecen al objeto (las esquinas del rectángulo, áreas vacías). La segmentación da el contorno exacto, lo que permite aplicaciones imposibles con detección: edición de fotos, conducción autónoma (saber dónde termina la calle), análisis médico (medir el área de una lesión), realidad aumentada, etc.

### El modelo: U-Net

Vamos a implementar **U-Net** (Ronneberger et al. 2015), una arquitectura que se diseñó originalmente para segmentación de imágenes biomédicas y que se convirtió en el caballito de batalla de la segmentación moderna. La idea central es muy simple:

- Un **camino contractivo** (encoder) que reduce la resolución y aumenta los canales — captura *qué* hay en la imagen.
- Un **camino expansivo** (decoder) que recupera la resolución original — produce el mapa de clases píxel por píxel.
- **Conexiones de salto** (skip connections) entre los dos caminos para no perder información espacial fina al bajar y al volver a subir.

La forma de "U" del diagrama (de ahí el nombre) refleja exactamente eso: bajar, dar la vuelta abajo, y subir mientras se "consultan" las activaciones del lado descendente.

![](https://miro.medium.com/max/720/1*YaLdptIoloK184uJQTH1HA.png)

> **Variante que vamos a implementar:** el paper original (2015) usa convoluciones 3×3 **sin padding**, lo que hace que cada doble convolución muerda 4 píxeles de los bordes. Resultado: el output sale con menor resolución que el input (388×388 desde 572×572), las skip connections necesitan recortes explícitos para alinearse, y la red se entrena solo sobre el centro de cada imagen. Esa variante "fiel al paper" tiene valor histórico pero es difícil de optimizar desde cero. Acá implementamos la **variante moderna** que usa padding=1 en cada conv 3×3 — la salida tiene exactamente el mismo tamaño que la entrada, las skips concatenan directamente y la red entrena sobre todos los píxeles. Es la receta que usan hoy nnU-Net, MONAI y `segmentation_models_pytorch`. Agregamos también **BatchNorm** después de cada convolución (tampoco estaba en el paper de 2015 — la técnica se popularizó después). Los demás elementos sí son los del paper: encoder de 5 niveles, decoder simétrico, skip connections en cada nivel.

### El dataset: Oxford-IIIT Pet

Vamos a usar **Oxford-IIIT Pet** (Parkhi et al. 2012), un dataset clásico de imágenes de mascotas. Cada imagen tiene asociada una máscara de segmentación **trimap** con tres valores: `1` para los píxeles de la mascota, `2` para el fondo y `3` para los píxeles de borde (esos los vamos a tratar como "ignorar" durante el entrenamiento porque el anotador no estaba seguro de a qué clase pertenecen). Para nuestro lab queda como un problema de segmentación **binaria**: clase 0 = fondo, clase 1 = mascota.

Pet es cómodo para un primer contacto con segmentación: tiene ~3.7k imágenes de trainval, las fotos están centradas en el sujeto, la ambigüedad de clase es baja y el dataset es lo suficientemente chico como para entrenar en una sesión de Colab. Eso permite enfocarnos en la arquitectura y el ciclo de entrenamiento sin que el cuello de botella sea la complejidad del dataset.

### Lo que vas a hacer

El laboratorio se divide en seis bloques:

1. **Sección A — Dataset:** explorar Oxford-IIIT Pet, entender el formato de los trimaps y armar los `DataLoader` de entrenamiento y validación.
2. **Sección B — Bloques de U-Net:** implementar las cinco piezas que componen la red — doble convolución, downsampling, upsampling, capa final y la función auxiliar de recorte para alinear skip connections.
3. **Sección C — Red completa:** ensamblar las piezas en la clase `UNet`.
4. **Sección D — Entrenamiento desde cero (prueba inicial):** entrenar la red sobre Pet por unas pocas epochs para confirmar que la arquitectura arranca y aprende algo. Usamos **pesos por clase** en la función de pérdida para no terminar con una red que predice mayormente "fondo". El resultado va a ser modesto a propósito — entrenar una U-Net desde cero sobre un dataset chico no es la receta de producción.
5. **Sección E — Predicción:** correr la red entrenada desde cero sobre imágenes nuevas y visualizar las máscaras predichas.
6. **Sección F — Fine-tuning desde una U-Net pre-entrenada:** repetir el entrenamiento partiendo de una U-Net cuyo encoder es un ResNet34 con pesos de ImageNet (vía `segmentation_models_pytorch`). En pocas epochs vamos a alcanzar resultados claramente mejores que entrenando desde cero. Esa es la receta real de producción y ésa es la moraleja del lab.

> **Importante — GPU y tiempo:** este laboratorio entrena dos redes profundas (la U-Net desde cero ~31M de parámetros + la U-Net con encoder ResNet34 ~24M) sobre imágenes de 256×256. **Activá la GPU en Colab** antes de empezar: *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)*. Cada entrenamiento toma **5-8 minutos** sobre T4. Total del lab: ~15 minutos de cómputo. Sin GPU es impracticable.
>
> Atención también con la memoria: las celdas de test al final de la Sección B crean modelos de juguete que pueden quedar referenciados después si no los liberás. Más adelante hay una celda de limpieza explícita justo antes del entrenamiento. Si igual ves errores de OOM (out of memory) durante el train, usá *Entorno de ejecución > Reiniciar* y ejecutá de nuevo todo de arriba a abajo.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- Para el material teórico (convolución transpuesta, FCN, segmentación semántica) consultá el notebook `Segmentacion.ipynb` de la clase.
- Las celdas de test al final de cada bloque te ayudan a verificar que tu implementación devuelve tensores con la forma correcta. **No las modifiques**: si fallan, el problema está en tu código.
- Los entrenamientos del Ej. 8 (prueba inicial desde cero) y del Ej. 10 (fine-tuning) toman **5-8 minutos cada uno** sobre Colab T4. Podés ir avanzando con otra cosa mientras corren.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas (enunciados, explicaciones, ejemplos provistos y el encabezado) **no se tocan**: la corrección se hace celda por celda de manera automática y modificar lo que no corresponde puede invalidar tu entrega.

Si necesitás probar algo fuera de una celda de actividad, hacelo en una copia aparte y revertí los cambios antes de entregar.

In [ ]:
# ─── Setup: imports y detección de GPU ──────────────────────────────────────
# Imports que usa el laboratorio de punta a punta. Acá no instalamos nada nuevo
# porque torch y torchvision ya vienen en Colab.
import os
import gc
import random
import tarfile
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms

device = (
    "cuda" if torch.cuda.is_available()        else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)
print(f"Versión de PyTorch: {torch.__version__}")
print(f"Dispositivo:        {device}")
if device == "cpu":
    print("ADVERTENCIA: sin GPU el entrenamiento va a ser inviable. "
          "Activá la GPU en Colab (T4) antes de continuar.")

In [ ]:
# ─── Setup: descarga del dataset Oxford-IIIT Pet ────────────────────────────
# Oxford-IIIT Pet (Parkhi et al. 2012) tiene ~7400 imágenes de mascotas (37
# razas de perros y gatos) con segmentación trimap: 1=mascota, 2=fondo, 3=borde.
# Bajamos los dos .tar.gz oficiales (~800 MB combinados, 1-2 minutos en Colab).
DATA_URL_IMG = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
DATA_URL_ANN = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"
DATA_DIR    = "/content/data/pet"
pet_dir     = DATA_DIR
images_dir  = os.path.join(pet_dir, "images")
trimaps_dir = os.path.join(pet_dir, "annotations", "trimaps")

os.makedirs(DATA_DIR, exist_ok=True)

def _download_and_extract(url, tag):
    tar_path = os.path.join(DATA_DIR, f"{tag}.tar.gz")
    print(f"Descargando {tag} desde {url}...")
    urllib.request.urlretrieve(url, tar_path)
    print(f"Extrayendo {tag}...")
    with tarfile.open(tar_path) as tar:
        tar.extractall(DATA_DIR)
    os.remove(tar_path)

if not os.path.isdir(images_dir):
    _download_and_extract(DATA_URL_IMG, "images")
if not os.path.isdir(trimaps_dir):
    _download_and_extract(DATA_URL_ANN, "annotations")

n_imgs = len([f for f in os.listdir(images_dir) if f.endswith(".jpg")])
n_msks = len([f for f in os.listdir(trimaps_dir) if f.endswith(".png")])
print(f"\npet_dir   = {pet_dir}")
print(f"Imágenes  = {n_imgs}")
print(f"Trimaps   = {n_msks}")

In [ ]:
# ─── Setup: clases, colormap y mapeo de trimap ──────────────────────────────
# Trimap original de Pet: 1=mascota, 2=fondo, 3=borde. Lo remapeamos a:
#   0 = background, 1 = pet, 255 = ignore (borde).
# Por qué borde como ignore: los bordes del trimap son una franja "no estoy
# seguro" del anotador y suelen ser ruidosos. Si los incluyéramos como una
# tercera clase, la red gastaría capacidad aprendiendo ruido.
PET_CLASSES  = ['background', 'pet']
PET_COLORMAP = [[0, 0, 0], [255, 100, 0]]   # negro / naranja

NUM_CLASSES  = len(PET_CLASSES)   # 2
IGNORE_INDEX = 255                # los píxeles de borde no se evalúan

print(f"Clases ({NUM_CLASSES}): {PET_CLASSES}")


# ─── Helper para visualización: leyenda con colores de las clases ──────────
# Se usa en el Ej. 9. Pega una fila de cuadritos coloreados arriba de la
# figura asociando cada color de PET_COLORMAP a su nombre de clase, más un
# cuadrito gris para los píxeles `ignore`.
from matplotlib.patches import Patch

def add_seg_legend(fig, class_names=PET_CLASSES, colormap=PET_COLORMAP):
    """Agrega una leyenda global a fig con los colores de las clases."""
    handles = [
        Patch(facecolor=tuple(c / 255 for c in colormap[i]),
              edgecolor='black', label=class_names[i])
        for i in range(len(class_names))
    ]
    handles.append(Patch(facecolor=(0.5, 0.5, 0.5),
                         edgecolor='black', label='ignore (borde)'))
    fig.legend(handles=handles, loc='upper center',
               ncol=len(handles), bbox_to_anchor=(0.5, 1.0))

In [ ]:
# ─── Setup: PetSegDataset con random crop + flip horizontal ─────────────────
# Esta clase está preescrita: armarla a mano excede el alcance del lab. Lo
# importante es que entiendas qué hace. Pasos clave en __getitem__:
#   1. Lee la imagen y el trimap.
#   2. Si la imagen es más chica que crop_size en alguna dimensión, le hace
#      un resize manteniendo aspect ratio (raro en Pet, pero hay un puñado
#      de imágenes muy chicas).
#   3. Random crop CONSISTENTE entre imagen y máscara — si recortara cada una
#      por su lado, los píxeles dejarían de coincidir.
#   4. Horizontal flip aleatorio (50% de probabilidad), también consistente.
#      Es la única augmentation que sumamos: multiplica efectivamente el
#      dataset sin costo.
#   5. Normaliza la imagen con la media/std de ImageNet.
#   6. Mapea el trimap {1=pet, 2=bg, 3=border} a {1, 0, 255} para que la
#      cross-entropy del Ej. 8 ignore los píxeles de borde con ignore_index.
#
# El parámetro `subset_size` permite tomar solo una porción del dataset.
# Lo usamos para que el train no demore más de ~20 minutos.
class PetSegDataset(torch.utils.data.Dataset):
    """Oxford-IIIT Pet para segmentación binaria (pet / background)."""

    def __init__(self, split, crop_size, pet_dir, subset_size=None, augment=True):
        self.crop_size = crop_size
        self.pet_dir   = pet_dir
        self.augment   = augment
        self.transform = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        # Lista de nombres del split (formato: "<name> <class_id> ...").
        txt = os.path.join(pet_dir, "annotations", f"{split}.txt")
        with open(txt) as f:
            names = [line.split()[0] for line in f
                     if line.strip() and not line.startswith("#")]
        self.names = names
        if subset_size is not None and subset_size < len(self.names):
            rng = random.Random(42)  # subset reproducible.
            self.names = rng.sample(self.names, subset_size)
        print(f"{split}: total={len(names)} usables={len(self.names)}"
              f"{' (subset)' if subset_size else ''}")

    def _read(self, name):
        img = Image.open(os.path.join(self.pet_dir, "images", f"{name}.jpg")
                         ).convert("RGB")
        msk = Image.open(os.path.join(self.pet_dir, "annotations",
                                       "trimaps", f"{name}.png"))
        # Si la imagen es más chica que el crop, resize al mínimo + un margen
        # para que después el random_crop pueda recortar sin error.
        if img.size[0] < self.crop_size[1] or img.size[1] < self.crop_size[0]:
            f = max(self.crop_size[1] / img.size[0],
                    self.crop_size[0] / img.size[1]) * 1.1
            new_size = (int(img.size[0] * f), int(img.size[1] * f))
            img = img.resize(new_size, Image.BILINEAR)
            msk = msk.resize(new_size, Image.NEAREST)
        return img, msk

    def __getitem__(self, idx):
        img, msk = self._read(self.names[idx])

        # Tensores: imagen (3, H, W) uint8, máscara (H, W) int64.
        img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1)
        msk_t = torch.from_numpy(np.array(msk, dtype=np.int64))

        # Random crop consistente entre imagen y máscara.
        i, j, h, w = transforms.RandomCrop.get_params(img_t, self.crop_size)
        img_t = transforms.functional.crop(img_t, i, j, h, w)
        msk_t = msk_t[i:i + h, j:j + w]

        # Horizontal flip (data augmentation simple, gratis y muy efectiva).
        if self.augment and random.random() < 0.5:
            img_t = transforms.functional.hflip(img_t)
            msk_t = torch.flip(msk_t, dims=[1])

        # Mapeo trimap {1=pet, 2=bg, 3=border} → {1=pet, 0=bg, 255=ignore}.
        out = torch.full_like(msk_t, IGNORE_INDEX)
        out[msk_t == 2] = 0
        out[msk_t == 1] = 1

        # Normalización imagen.
        img_norm = self.transform(img_t.float() / 255)
        return img_norm, out

    def __len__(self):
        return len(self.names)

---
## Sección A: El dataset Oxford-IIIT Pet

Antes de meternos con la red, conviene pasar un rato mirando los datos. En segmentación las etiquetas son imágenes y eso obliga a pensar cosas que no aparecen en clasificación: **cómo se interpreta cada píxel del trimap** y **cómo se aumentan/recortan las imágenes manteniendo la coincidencia entre imagen y máscara**.

### Por qué no usamos `Resize` en segmentación

En clasificación uno hace `transforms.Resize((224, 224))` y listo: la imagen se reescala a un tamaño fijo y la red la procesa. En **segmentación no se puede hacer eso** sin pensarlo dos veces. Si reescalo la imagen, ¿qué hago con la máscara?

- Si la reescalo con interpolación bilineal (la default), los valores de los píxeles dejan de ser índices de clase enteros y pasan a ser promedios ponderados — no tiene sentido decir "este píxel es 17.3" cuando los índices son discretos.
- Si la reescalo con interpolación *nearest neighbor*, mantengo los índices pero pierdo precisión en los bordes de los objetos: el contorno se vuelve dentado y poco fiel.
- Si reescalo a un tamaño distinto al original, además tengo que ser cuidadoso de mantener la **misma transformación** entre imagen y máscara, o pierden la correspondencia píxel a píxel.

La solución estándar — la que usa nuestro `PetSegDataset` — es **recortar (crop)** un parche de tamaño fijo en lugar de reescalar. El recorte aleatorio durante el entrenamiento cumple además el rol de data augmentation: cada época ve un parche distinto de cada imagen.

### Una mirada rápida al trimap

Antes de pasar a los `DataLoader` conviene mirar a ojo qué forma tienen las imágenes y qué información codifica el trimap. La función `read_pet_images` que aparece debajo lee unas pocas imágenes y sus máscaras directamente del disco — sin pasar por `PetSegDataset` — usando la lista de nombres de archivo del split correspondiente (`annotations/trainval.txt` o `annotations/test.txt`, donde cada línea tiene la forma `<nombre> <class_id> <species> <breed_id>` y solo nos interesa el primer campo). La definimos acá porque más adelante (Ej. 8 y Ej. 9) la vamos a reusar para tomar imágenes del split de test y compararlas con las predicciones del modelo.

Para leer los archivos usamos `torchvision.io.read_image`, que devuelve directamente un tensor `uint8`. Los trimaps **no** requieren conversión a RGB: vienen como paleta indexada de un solo canal con valores `{1, 2, 3}`, así que la lectura por default ya entrega exactamente lo que necesitamos. Para visualizarlos pasamos a `imshow` un colormap discreto (`cmap='viridis'`) — si los mostráramos con un colormap continuo, los tres valores se verían casi negros.

La grilla resultante muestra, en su fila superior, las primeras 4 imágenes de `trainval`, y en la fila inferior los trimaps correspondientes. Hay dos cosas para fijarse:

- **Los trimaps tienen tres valores únicos**: `1` para los píxeles de la mascota, `2` para el fondo y `3` para los bordes "no estoy seguro" del anotador (que después tratamos como `IGNORE_INDEX`).
- **El fondo domina la imagen** aunque la cámara esté apuntando claramente al animal. Una mascota típica ocupa el 25-35% del cuadro; el resto es piso, pasto, sofá, pared. Sumado sobre el dataset, el fondo se lleva ~70% de los píxeles y la mascota ~30%. Es un desbalance suave pero claro y vamos a tener que tenerlo en cuenta cuando entrenemos en la **Sección D** (ahí calculamos pesos por clase para que la red no colapse a "predecir todo fondo").

In [ ]:
# ─── read_pet_images: lectura directa desde el disco ────────────────────────
# Lee las primeras `n` imágenes y trimaps del split indicado, sin pasar por
# PetSegDataset. La reusamos en los Ej. 8 y 9 para comparar predicciones con
# el ground truth.
def read_pet_images(pet_dir, n, split='trainval'):
    """
    Lee las primeras n imágenes y trimaps del split indicado.

    Retorna:
    features (list[Tensor]): imágenes RGB uint8 (3, H, W).
    labels   (list[Tensor]): trimaps uint8 (1, H, W) con valores {1, 2, 3}.
    """
    txt = os.path.join(pet_dir, 'annotations', f'{split}.txt')
    with open(txt) as f:
        # Cada línea: <nombre> <class_id> <species> <breed_id>. Tomamos el primero.
        names = [line.split()[0] for line in f
                 if line.strip() and not line.startswith('#')]
    features, labels = [], []
    for name in names[:n]:
        features.append(torchvision.io.read_image(
            os.path.join(pet_dir, 'images', f'{name}.jpg')))
        # Los trimaps son grayscale (1 canal); la lectura default ya los
        # entrega como (1, H, W) con uint8 valores {1, 2, 3}.
        labels.append(torchvision.io.read_image(
            os.path.join(pet_dir, 'annotations', 'trimaps', f'{name}.png')))
    return features, labels


# ─── Visualización: 4 imágenes y sus trimaps ───────────────────────────────
# matplotlib espera (H, W, C); los tensores de PyTorch vienen como (C, H, W),
# así que reordenamos con permute(1, 2, 0) antes de mostrar las imágenes.
n = 4
imgs, masks = read_pet_images(pet_dir, n, split='trainval')

fig, axs = plt.subplots(2, n, figsize=(4 * n, 8))
for i in range(n):
    axs[0, i].imshow(imgs[i].permute(1, 2, 0))
    axs[0, i].set_title(f'Imagen {i}')
    axs[0, i].axis('off')
    # masks[i] tiene shape (1, H, W). Mostramos el único canal con un colormap
    # discreto que distinga los tres valores del trimap.
    axs[1, i].imshow(masks[i][0], cmap='viridis')
    axs[1, i].set_title(f'Trimap {i}  (vals: {torch.unique(masks[i]).tolist()})')
    axs[1, i].axis('off')
plt.tight_layout()
plt.show()

### Ejercicio 1 — Instanciar `PetSegDataset` y armar los `DataLoader`

**Objetivo:** Usar la clase `PetSegDataset` (preescrita) para crear los datasets de train y val, envolverlos en `DataLoader` e inspeccionar la forma de un batch.

**Enunciado:**

1. Definí el tamaño de crop como una tupla `(256, 256)`. Este tamaño no es arbitrario: la U-Net que vamos a implementar después baja la resolución 4 veces dividiendo por 2 cada vez, así que el input necesita ser **divisible por `2^4 = 16`** para que las cuentas cierren. 256 es la potencia de 2 más cómoda en este rango; la justificación detallada aparece en la sección C.
2. Creá dos datasets:
   - **train:** instanciá `PetSegDataset` sobre el split `'trainval'` con un `subset_size` de 1500 imágenes y data augmentation activada. El subset es por una razón puramente práctica: Pet tiene ~3700 imágenes en trainval y entrenar con todas tomaría más de media hora; con 1500 alcanzamos `val_acc` razonable en ~20 minutos.
   - **val:** instanciá `PetSegDataset` sobre el split `'test'` con todo el dataset (sin subset) y la augmentation desactivada — en validación no queremos que cada época vea una versión distinta de la misma imagen.
3. Envolvé cada dataset en un `DataLoader` (`torch.utils.data.DataLoader`) con tamaño de batch 8, descartando el último batch si queda incompleto y con un par de workers para paralelizar la lectura del disco. Acordate de que el train se baraja entre épocas y val no.
4. Pedile al iterador de train su primer batch e imprimí la forma de las imágenes y las máscaras, junto con el rango de valores de las máscaras (mínimo y máximo). Las imágenes deberían tener forma `(8, 3, 256, 256)` con valores normalizados (no en [0, 1]); las máscaras `(8, 256, 256)` con valores en `{0, 1, 255}` (fondo, mascota, ignorar).

> **Pista:** Descartar el último batch incompleto se logra con un argumento del `DataLoader` cuyo nombre habla por sí solo. En segmentación no es crítico, pero es la convención.
>
> **Nota:** Usar 2 workers paraleliza la carga del disco. En Colab, valores muy altos (≥4) a veces traen más overhead que beneficio.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

El batch de imágenes tiene shape `(8, 3, 256, 256)` y el batch de máscaras tiene shape `(8, 256, 256)` — sin la dimensión de canales. ¿Por qué la máscara no tiene canales? ¿Qué está representando cada valor del tensor `Y`, y qué significa específicamente el valor `255`?

*(Escribí tu respuesta acá)*

---
## Sección B: Bloques de la U-Net

Vamos a construir la U-Net armando primero las piezas de Lego que la componen. La idea es que cada bloque sea autocontenido y testeable: una vez que cada uno pasa su test, ensamblar la red completa es casi mecánico.

Las piezas son cinco:

1. **`SimpleConvolution`** — el bloque de **doble convolución** que se repite en cada nivel de la U. Cada convolución 3×3 va con `padding=1` para preservar la resolución espacial.
2. **`DownConvolution`** — el bloque del lado descendente: maxpool + doble conv. Reduce la resolución espacial a la mitad (vía maxpool) y aumenta los canales (vía doble conv).
3. **`UpConvolution`** — el bloque del lado ascendente: doble conv + convolución transpuesta. Procesa las features que llegan concatenadas y duplica la resolución para subir un nivel.
4. **`LastConvolution`** — el bloque final: doble conv + convolución 1×1 que mapea de los 64 canales que llegan a las `num_classes` de salida.
5. **`crop_img`** — función auxiliar que recorta un tensor para que coincida espacialmente con otro. En esta versión moderna de la U-Net **no la usamos dentro de la red** (las dimensiones cuadran solas gracias al padding), pero la implementamos como ejercicio: en la U-Net del paper original, donde las convs no tienen padding, esta operación es indispensable para alinear los skips antes de concatenar. Es un buen ejercicio sobre slicing de tensores 4D y entender por qué la versión moderna se simplifica.

> **Nota sobre el padding:** las convoluciones 3×3 con `padding=1` preservan la resolución espacial — el output de la doble convolución tiene exactamente la misma forma que el input. La consecuencia para las skip connections es importante: como cada nivel de la bajada y el correspondiente nivel de la subida tienen exactamente la misma forma espacial, los podemos concatenar **directamente**, sin recortar nada. Eso es una simplificación significativa respecto del paper original (Ronneberger 2015), donde cada doble convolución sin padding mordía 4 píxeles y los skips del lado descendente terminaban siendo más grandes que los del ascendente — había que recortarlos al centro antes de concatenar. La variante moderna con padding (la que vamos a implementar) es la que se usa hoy en producción.

### Ejercicio 2 — `SimpleConvolution`: bloque de doble convolución

**Objetivo:** Implementar el bloque que aparece en cada nivel de la U-Net: dos convoluciones 3×3 con ReLU intercaladas, más un dropout suave al final.

![](https://miro.medium.com/max/640/1*Uan1yYCi3ZO1xrtLohyWzg.png)

**Enunciado:**

Implementá una clase `SimpleConvolution` (subclase de `nn.Module`) cuyo constructor reciba la cantidad de canales de entrada y la cantidad de canales de salida. El bloque tiene que aplicar, en orden:

1. Una primera convolución 3×3 (`nn.Conv2d`) **con padding=1** que mapee de los canales de entrada a los de salida, seguida de una BatchNorm 2D (`nn.BatchNorm2d`) sobre los canales de salida y una ReLU (`nn.ReLU`).
2. Una segunda convolución 3×3 también con padding=1 que mantenga la cantidad de canales (entrada y salida igual a los canales de salida del paso anterior), seguida de otra `nn.BatchNorm2d` y otra `nn.ReLU`.
3. Un dropout (`nn.Dropout`) con probabilidad 0.1 al final.

Para una entrada de forma `(B, c_in, H, W)`, la salida tiene que ser `(B, c_out, H, W)` — el padding=1 hace que cada convolución 3×3 preserve exactamente la resolución espacial. La BatchNorm, la ReLU y el Dropout tampoco cambian la forma del tensor.

> **Pista:** Podés guardar la pila de capas en un `nn.Sequential` dentro de `__init__` y que `forward` sea casi trivial.

> **Nota sobre BatchNorm:** la U-Net del paper original (2015) **no usa BatchNorm** — la técnica recién se popularizó después. Sin BN una U-Net entrenada **desde cero** sobre datasets chicos cuesta mucho convergir: los gradientes se atenúan en redes profundas y la red termina en mínimos triviales (predecir siempre la clase mayoritaria). Hoy todas las implementaciones modernas incluyen BatchNorm después de cada Conv2d. Lo agregamos por la misma razón.

In [ ]:
# Tu código aquí

In [ ]:
# ─── Test SimpleConvolution ────────────────────────────────────────────────
# Input chico (1x1x32x32) para que el test apenas use memoria.
# Con padding=1 las convs 3x3 preservan la resolución espacial: 32x32 → 32x32.
block = SimpleConvolution(1, 16)
inp = torch.rand(1, 1, 32, 32)
out = block(inp)
assert out.shape == (1, 16, 32, 32), f"Forma incorrecta: {tuple(out.shape)}"
print("Test SimpleConvolution OK.")
del block, inp, out

### Ejercicio 3 — `DownConvolution`: bajar un nivel

**Objetivo:** Implementar el bloque del camino descendente: primero un maxpool 2×2 que reduce la resolución a la mitad, después una `SimpleConvolution` que actualiza los canales.

![](https://miro.medium.com/max/640/1*9zoULdYOeKQsLWQGExhVlQ.png)

**Enunciado:**

Implementá una clase `DownConvolution` (subclase de `nn.Module`) cuyo constructor reciba canales de entrada y de salida. El bloque tiene que aplicar, en orden:

1. Un max-pooling 2×2 (`nn.MaxPool2d`) con stride 2 (divide a la mitad alto y ancho).
2. El bloque de doble convolución del ejercicio anterior, mapeando de los canales de entrada a los de salida.

Para una entrada de forma `(B, c_in, H, W)` con `H` y `W` pares, la salida tiene que ser `(B, c_out, H/2, W/2)` — la única reducción espacial viene del maxpool, porque la doble convolución preserva resolución.

> **Pista:** Podés reutilizar la `SimpleConvolution` que acabás de definir guardándola como un atributo del módulo. No hace falta reimplementarla.

In [ ]:
# Tu código aquí

In [ ]:
# ─── Test DownConvolution ──────────────────────────────────────────────────
block = DownConvolution(16, 32)
inp = torch.rand(1, 16, 32, 32)
out = block(inp)
# 32 → maxpool → 16; simpleconv preserva → 16. canales 16 → 32.
assert out.shape == (1, 32, 16, 16), f"Forma incorrecta: {tuple(out.shape)}"
print("Test DownConvolution OK.")
del block, inp, out

### Ejercicio 4 — `UpConvolution`: subir un nivel

**Objetivo:** Implementar el bloque del camino ascendente: una `SimpleConvolution` que procesa las features que llegan ya concatenadas, seguida de una **convolución transpuesta** que duplica la resolución espacial.

![](https://miro.medium.com/max/640/1*nmfwdmaW5A7_zxI0BcPcGQ.png)

**Enunciado:**

Implementá una clase `UpConvolution` (subclase de `nn.Module`) cuyo constructor reciba canales de entrada y de salida. El bloque tiene que aplicar, en orden:

1. La doble convolución del Ej. 2, que recibe el tensor concatenado (skip + nivel inferior) con los canales de entrada y los baja a los canales de salida. Como la doble convolución preserva la resolución espacial, la salida tiene el mismo alto y ancho que la entrada.
2. Una convolución transpuesta (`nn.ConvTranspose2d`) con kernel 2×2 y stride 2 que duplique la resolución espacial y, además, **reduzca los canales a la mitad** respecto de la salida del paso anterior. La razón de bajar los canales acá: a la salida de este bloque concatenamos con una skip connection del nivel superior que ya tiene esa cantidad de canales — para que al concatenar quede una cantidad redonda (skip + ascendente, cada uno con la mitad), conviene que el ascendente venga ya con la mitad.

Para una entrada de forma `(B, c_in, H, W)`, la salida tiene que ser `(B, c_out // 2, 2H, 2W)` — la doble convolución preserva la resolución y la convolución transpuesta con stride 2 la duplica.

> **Pista:** La convolución transpuesta con kernel 2×2 y stride 2 es la inversa "de tamaño" de un MaxPool 2×2: por cada píxel de entrada produce un parche 2×2 en la salida. Repasá la sección sobre convolución transpuesta del notebook teórico si querés volver a ver cómo opera.

In [ ]:
# Tu código aquí

In [ ]:
# ─── Test UpConvolution ────────────────────────────────────────────────────
block = UpConvolution(64, 32)
inp = torch.rand(1, 64, 16, 16)
# 16 → simpleconv preserva → 16 → trans conv x2 → 32
# canales: 64 → 32 (simpleconv) → 16 (trans conv halves)
out = block(inp)
assert out.shape == (1, 16, 32, 32), f"Forma incorrecta: {tuple(out.shape)}"
print("Test UpConvolution OK.")
del block, inp, out

### Ejercicio 5 — `LastConvolution`: bloque final

**Objetivo:** Implementar el bloque final que cierra la U-Net: `SimpleConvolution` que termina de procesar las features y una **convolución 1×1** que mapea a `num_classes` canales (uno por clase semántica).

![](https://miro.medium.com/max/720/1*cqs5XJRsBXS0RAkdIl_wUQ.png)

**Enunciado:**

Implementá una clase `LastConvolution` (subclase de `nn.Module`) cuyo constructor reciba **tres** parámetros: canales de entrada, canales intermedios y número de clases. El bloque tiene que aplicar, en orden:

1. La doble convolución del Ej. 2, que baja los canales de entrada a los canales intermedios (típicamente 64) y preserva la resolución espacial.
2. Una convolución 1×1 (`nn.Conv2d` con `kernel_size=1`) que mezcle esos canales intermedios y produzca un canal por clase. Recordá que la convolución 1×1 tampoco cambia la resolución espacial: actúa como una capa lineal por píxel sobre la dimensión de canales.

Para una entrada de forma `(B, c_in, H, W)`, la salida tiene que ser `(B, num_classes, H, W)` — el alto y ancho se preservan en todo el bloque.

> **Pista:** La convolución 1×1 sobre features con `c_int` canales y salida `num_classes` equivale a aplicar la misma matriz `(num_classes, c_int)` a cada vector de features, posición por posición. Es un buen ejercicio mental verificar por qué eso es lo mismo que una capa lineal píxel a píxel.

In [ ]:
# Tu código aquí

In [ ]:
# ─── Test LastConvolution ──────────────────────────────────────────────────
block = LastConvolution(32, 16, num_classes=3)
inp = torch.rand(1, 32, 32, 32)
out = block(inp)
# Resolución se preserva en todo el bloque: 32 → 32.
# canales: 32 → 16 (simpleconv) → 3 (1x1)
assert out.shape == (1, 3, 32, 32), f"Forma incorrecta: {tuple(out.shape)}"
print("Test LastConvolution OK.")
del block, inp, out

### Ejercicio 6 — `crop_img`: alinear tensores espacialmente

**Objetivo:** Implementar una función auxiliar que recorta un tensor en el centro para que coincida espacialmente con otro.

![](https://miro.medium.com/max/720/1*2XyH7YGv7MuJWPycqx7hew.png)

**Contexto:**

En la U-Net **del paper original** (Ronneberger 2015), las convoluciones 3×3 son **sin padding** y van mordiendo píxeles del borde a medida que avanzan. Eso hace que los feature maps del lado descendente terminen siendo **más grandes** espacialmente que los del ascendente con los que se concatenan. Antes de concatenar hay que **recortar el centro** del tensor más grande para que las dos formas espaciales coincidan exactamente.

Nuestra U-Net moderna con padding=1 **no necesita esta operación dentro de la red** (las convs preservan resolución y los skips cuadran solos). Igual implementamos `crop_img` por su valor pedagógico:

- En la U-Net del paper original, **es indispensable**: cada concatenación entre el lado descendente y el ascendente requiere un center-crop del lado más grande. Saber cómo se hace es entender por qué la variante moderna con padding es una simplificación significativa.
- Es un buen ejercicio sobre slicing de tensores 4D y manejo simétrico de bordes — operaciones que aparecen seguido cuando se trabaja con redes de visión.

**Enunciado:**

Implementá una función `crop_img` que reciba dos tensores 4D — el "fuente" (más grande, el que se recorta) y el "objetivo" (el que dicta la forma espacial final). Asumí que ambos tienen forma `(B, C, H, W)` con `B` y los canales eventualmente distintos.

Pasos sugeridos:

1. Calculá la diferencia entre el alto del fuente y el del objetivo (idem para el ancho).
2. Repartí esa diferencia simétricamente entre el borde de arriba y el de abajo (idem izquierda/derecha). En el caso de diferencia impar, no es crítico cómo resolvas el píxel suelto siempre que el resultado tenga la forma esperada.
3. Devolvé el fuente con esos bordes recortados.

> **Pista:** Indexar tensores con slices te deja extraer un parche directo: `tensor[:, :, top:bottom, left:right]`. No hace falta ningún `for`.

In [ ]:
# Tu código aquí

In [ ]:
# ─── Test crop_img ─────────────────────────────────────────────────────────
src = torch.rand(1, 16, 32, 32)
tgt = torch.rand(1, 8, 20, 20)
cropped = crop_img(src, tgt)
assert cropped.shape == (1, 16, 20, 20), f"Forma incorrecta: {tuple(cropped.shape)}"
# Chequeo de "centrado": el contenido del recorte debe coincidir con el
# parche central de src.
assert torch.allclose(cropped, src[:, :, 6:26, 6:26])
print("Test crop_img OK.")
del src, tgt, cropped

---
## Sección C: Ensamblar la U-Net completa

Con los cinco bloques listos, armar la red es seguir el diagrama. La forma de "U" del modelo se ve directamente en el código: dos listas paralelas de bloques (uno de bajada y uno de subida) que se conectan por las skip connections.

### Recorrido de formas con `crop_size = (256, 256)`

Antes de implementar conviene ver cómo varían las formas espaciales y de canales a lo largo de la red, para que el código tenga sentido. Con un input de `(B, 3, 256, 256)` (donde `NC = NUM_CLASSES`):

| Paso | Bloque | Forma de salida | Comentario |
|---|---|---|---|
| 0 | input | `(3, 256, 256)` | imagen normalizada |
| 1 | `SimpleConvolution(3, 64)` | `(64, 256, 256)` | **skip1** — preserva resolución |
| 2 | `DownConvolution(64, 128)` | `(128, 128, 128)` | **skip2** — pool ÷2 |
| 3 | `DownConvolution(128, 256)` | `(256, 64, 64)` | **skip3** |
| 4 | `DownConvolution(256, 512)` | `(512, 32, 32)` | **skip4** |
| 5 | `DownConvolution(512, 1024)` | `(1024, 16, 16)` | fondo de la U |
| 6 | `ConvTranspose2d(1024, 512)` | `(512, 32, 32)` | bridge: sube un nivel |
| 7 | concat con **skip4** | `(1024, 32, 32)` | 512 + 512 |
| 8 | `UpConvolution(1024, 512)` | `(256, 64, 64)` | conv preserva → trans conv ×2 |
| 9 | concat con **skip3** | `(512, 64, 64)` | 256 + 256 |
| 10 | `UpConvolution(512, 256)` | `(128, 128, 128)` | |
| 11 | concat con **skip2** | `(256, 128, 128)` | 128 + 128 |
| 12 | `UpConvolution(256, 128)` | `(64, 256, 256)` | |
| 13 | concat con **skip1** | `(128, 256, 256)` | 64 + 64 |
| 14 | `LastConvolution(128, 64, NC)` | `(NC, 256, 256)` | output |

La salida es `(NC, 256, 256)`: por cada píxel del input, `NC` logits que el `argmax` colapsa a la clase predicha. **El output tiene exactamente la misma resolución que el input**, así que la red supervisa cada píxel — no se pierde nada en los bordes. Esa es la ganancia operativa de usar `padding=1` en las convoluciones.

> **Por qué `crop_size = 256`:** elegimos un tamaño que sea divisible por 16 (=`2^4`) para que las cuatro divisiones por 2 del lado descendente cierren todas en enteros. 256 = 2^8 es la potencia de 2 más cómoda en este rango: al bajar 4 veces llega a 16 y al subir vuelve a 256. Si elegiéramos un número como 257 (impar), el primer pool daría 128 con un píxel de pérdida, después 64, después 32, después 16 (perdiendo medio píxel por nivel) y al subir no podríamos reconstruir 257 exactamente — habría desencuentros entre skips. Con `padding=1` la simetría se mantiene solo si el tamaño de entrada es divisible por la cantidad de niveles de pool.

> **Atención al primer paso ascendente:** después del fondo de la U (`DownConvolution(512, 1024)`) la red **necesita un upsampling adicional ANTES** del primer `UpConvolution`. Lo hacemos con una `ConvTranspose2d(1024, 512, kernel_size=2, stride=2)` independiente, que duplica la resolución (16 → 32) y baja los canales a la mitad para que al concatenar con `skip4` (que tiene 512 canales) dé `1024 = 512 + 512` canales para el primer `UpConvolution(1024, 512)`.

### Ejercicio 7 — Clase `UNet`

**Objetivo:** Ensamblar todos los bloques en la red completa.

**Enunciado:**

Implementá una clase `UNet` (subclase de `nn.Module`) cuyo constructor reciba la cantidad de canales de entrada y la cantidad de clases. Tu trabajo es ensamblar los bloques que ya tenés siguiendo la tabla de shapes de la sección C: para cada transición de la tabla, el bloque que la cumple es siempre uno de los que ya implementaste.

Para que no te pierdas en el armado, la celda de abajo viene con **el esqueleto de la clase** ya escrito: `__init__` y `forward` están separados en bloques numerados, con los nombres de los submódulos sugeridos (`self.start`, `self.down1..4`, `self.bridge`, `self.up1..3`, `self.last`) y las variables de las skip connections (`skip1..skip4`) ya elegidas. Tu trabajo es completar las llamadas concretas dentro de cada bloque mirando la tabla de shapes.

**En el `__init__`** vas a declarar:

- El bloque inicial que va antes del primer pooling. Es el único que no tiene maxpool delante y mapea de los canales de entrada de la imagen a 64 canales.
- Cuatro bloques descendentes, uno por nivel del encoder. La tabla te dice los pares de canales de entrada/salida en cada uno (la regla simple: en cada bajada se duplican los canales).
- El upsampling intermedio entre el fondo de la U y el primer bloque ascendente: la operación que duplica la resolución espacial y baja los canales de 1024 a 512. Se puede armar con una sola convolución transpuesta (`nn.ConvTranspose2d`) de kernel 2×2 y stride 2 (no hace falta envolverla en un módulo propio).
- Tres bloques ascendentes, uno por cada nivel del decoder. Mirá la tabla para deducir qué canales tiene cada uno a la entrada y a la salida — recordá que cada UpConv recibe un tensor concatenado (lado descendente + lado ascendente) y entrega un tensor con la mitad de los canales de salida intermedios, listo para concatenar de nuevo arriba.
- El bloque final, que mapea a `num_classes` canales.

**En el `forward`** seguí el diagrama de la U:

1. Aplicá el bloque inicial y los cuatro descendentes, guardando la salida de los **cuatro primeros niveles** como skip connections (las vas a usar más adelante para concatenar). El quinto nivel — el fondo de la U — no necesita skip.
2. Aplicá el upsampling intermedio.
3. Concatená con la skip del nivel correspondiente y aplicá el siguiente bloque ascendente. Repetí tres veces y cerrá con el bloque final. Como las convs preservan resolución (gracias al `padding=1`), las skips y los feature maps ascendentes ya cuadran espacialmente — concatenan directamente, sin recortar.

> **Pista:** Para concatenar una lista de tensores sobre un eje usá `torch.cat([...], dim=...)`. Para una pila `(B, C, H, W)` el eje de canales es `dim=1`.

In [ ]:
class UNet(nn.Module):
    """
    U-Net moderna (variante con padding=1 + BatchNorm).

    Entrada: (B, input_channel, 256, 256)  → Salida: (B, num_classes, 256, 256)
    """
    def __init__(self, input_channel, num_classes):
        super().__init__()
        # ─── Encoder: bloque inicial + 4 DownConvolution ────────────────────
        # Mirá la tabla de shapes (sección C) para los pares (in, out).
        # Nombres sugeridos: self.start, self.down1, self.down2, self.down3, self.down4
        # Tu código aquí

        # ─── Bridge: ConvTranspose2d entre el fondo y el primer UpConvolution
        # Sube la resolución de (1024, 16, 16) a (512, 32, 32) usando
        # kernel_size=2 y stride=2. Nombre sugerido: self.bridge
        # Tu código aquí

        # ─── Decoder: 3 UpConvolution + bloque final ────────────────────────
        # Mirá la tabla de shapes para los pares (in, out) de cada UpConv.
        # Recordá que cada UpConv recibe un tensor concatenado (skip + abajo).
        # Nombres sugeridos: self.up1, self.up2, self.up3, self.last
        # Tu código aquí

    def forward(self, x):
        # ─── Bajada: aplicá los 5 bloques del encoder ───────────────────────
        # Guardá las salidas de los CUATRO PRIMEROS niveles como skip
        # connections — las vas a usar al subir. El quinto nivel (down4) NO
        # necesita skip: es el fondo de la U.
        # skip1 = ...                # salida de self.start
        # skip2 = ...                # salida de self.down1
        # skip3 = ...                # salida de self.down2
        # skip4 = ...                # salida de self.down3
        # x     = ...                # salida de self.down4 (fondo de la U)
        # Tu código aquí

        # ─── Subida: bridge + 3 UpConvolution + bloque final ────────────────
        # Para concatenar usá torch.cat([..., ...], dim=1).
        # Mirá la tabla: en cada paso ascendente concatenás con la skip del
        # mismo nivel ANTES de pasar por el UpConvolution / LastConvolution.
        # x = self.bridge(x)
        # x = self.up1(...)          # concat con skip4
        # x = self.up2(...)          # concat con skip3
        # x = self.up3(...)          # concat con skip2
        # x = self.last(...)         # concat con skip1
        # Tu código aquí

        return x

In [ ]:
# ─── Test UNet ─────────────────────────────────────────────────────────────
# Test con num_classes=3 e input chico para no quemar memoria. Verificamos
# que la forma de salida es la misma que la de entrada — esa es la propiedad
# clave de la U-Net con padding=1.
unet_test = UNet(input_channel=3, num_classes=3)
inp = torch.rand(1, 3, 256, 256)
with torch.no_grad():
    out = unet_test(inp)
print(f"input  : {tuple(inp.shape)}")
print(f"output : {tuple(out.shape)}  (esperado: (1, 3, 256, 256))")
assert out.shape == (1, 3, 256, 256), f"Forma incorrecta: {tuple(out.shape)}"
print("Test UNet OK.")
del unet_test, inp, out

**Pregunta de análisis:**

La U-Net que implementaste tiene dos diferencias clave respecto de la del paper original (Ronneberger 2015): nuestras convoluciones 3×3 usan `padding=1` (las del paper no tenían padding), y agregamos `BatchNorm2d` después de cada `Conv2d` (el paper no la usaba). Para cada uno de esos cambios, indicá qué consecuencia tiene en la forma del output (input 572×572 vs nuestro 256×256) y/o en el comportamiento del entrenamiento.

*(Escribí tu respuesta acá)*

---
## Sección D: Entrenamiento

Antes de empezar a entrenar, hay dos cosas que solemos pasar por alto y que importan particularmente en segmentación:

1. **Liberar memoria.** Las celdas de test crearon tensores y modelos pequeños que pueden quedar referenciados todavía. En GPU eso se nota porque cuando creamos la U-Net "real" puede aparecer un OOM si la suma del modelo nuevo + lo que quedó de los tests se pasa de la VRAM de la T4 (~15 GB).
2. **Pesos por clase.** Como ya señalamos al mirar los trimaps al inicio de la sección A, el fondo domina el dataset (~70% de los píxeles). Sin pesos, la red converge a "todo es fondo". Vamos a calcular los pesos como `1 / frecuencia_de_clase` y pasarlos a la cross-entropy.

In [ ]:
# ─── Limpieza de memoria antes de entrenar ──────────────────────────────────
# Las celdas de test crearon objetos que pueden quedar referenciados. Forzamos
# garbage collection y vaciamos la cache de CUDA. Si no querés tener que pensar
# en esto, lo equivalente es Entorno de ejecución > Reiniciar y ejecutar todo:
# pero en general conviene saber que existen estas dos llamadas.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM libre tras limpiar: "
          f"{torch.cuda.mem_get_info()[0] / 1e9:.2f} GB de "
          f"{torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

### Cálculo de pesos por clase

La receta es:

1. Recorrer todo el split de train contando cuántos píxeles hay de cada clase (los píxeles `IGNORE_INDEX=255` no se cuentan).
2. Calcular la frecuencia relativa `freq[c] = pixels_clase_c / pixels_totales`.
3. Definir `weights[c] = 1 / sqrt(freq[c] + ε)` con un `ε` pequeño para evitar dividir por 0 si una clase no aparece.
4. **Normalizar** los pesos para que sumen `num_classes`. Eso mantiene el orden de magnitud de la pérdida estable: si los pesos son enormes, la pérdida también lo es y el learning rate adecuado cambia.

Lo usamos pasándole el vector de pesos a `nn.CrossEntropyLoss(weight=weights, ignore_index=255)`.

> **Por qué `1/√freq` y no `1/freq` directamente:** `1/freq` parece la opción "natural" pero produce pesos demasiado desbalanceados. Si el fondo es 10× más frecuente que la mascota, con `1/freq` los pesos quedan en ratio 10:1 — la red descubre rápido que conviene **nunca** predecir fondo (un error en un píxel minoritario cuesta diez veces más) y termina con accuracy de píxel peor que la baseline trivial. La **raíz cuadrada** comprime ese rango: el mismo ratio 10× de frecuencias da pesos 3.16:1, agresivos pero no tóxicos. En nuestro caso (Pet, fondo ~0.70 / mascota ~0.30) el ratio max/min entre pesos pasa de ~2.3× con `1/freq` a ~1.5× con `1/√freq`. Es la receta que usan, por ejemplo, ENet (Paszke et al. 2016) y SegNet (Badrinarayanan et al. 2017) en sus papers de segmentación.

### Ejercicio 8 — Entrenamiento de la U-Net (prueba inicial desde cero)

**Objetivo:** Calcular los pesos por clase, definir la función de pérdida ponderada y entrenar la U-Net sobre Pet **desde cero** (con pesos inicializados al azar) por unas pocas epochs. La idea es confirmar que la arquitectura arranca y produce predicciones útiles — no obtener el mejor modelo posible. En el Ej. 10 vamos a repetir el experimento partiendo de un encoder pre-entrenado y comparar resultados.

> ⚠️ **Atención al tiempo de entrenamiento:** este ejercicio entrena la U-Net por **4 epochs** sobre las 1500 imágenes del subset. En **GPU T4 de Colab tarda aproximadamente 5-7 minutos**.
>
> ¿Por qué solo 4 epochs y no 12 o 20? Porque el cuello de botella **no es la cantidad de iteraciones**: una U-Net de 31M de parámetros entrenada desde cero sobre 1500 imágenes sigue convergiendo lento aunque le des una hora. Con 4 epochs ya vemos la tendencia (sale del baseline trivial de "todo fondo") y nos sobra tiempo para el fine-tuning del Ej. 10, que es donde realmente se ve qué saca al modelo del estancamiento.

**Enunciado:**

Esta es la celda principal de entrenamiento desde cero. Tiene varias partes; el código tiene la estructura armada con bloques numerados, completá los huecos donde dice `# Tu código aquí`.

1. **Conteo de píxeles por clase (PRE-RESUELTO):** la celda viene con esta parte ya implementada — recorre `pet_train`, acumula cuántos píxeles tiene cada clase descartando los `IGNORE_INDEX`, y arma una pequeña tabla de frecuencias. Tarda 1-2 minutos la primera vez. No hay nada que completar; pasá a la parte 2 cuando termine de imprimir.
2. **Pesos:** la celda viene con las dos primeras líneas pre-resueltas (cálculo de la frecuencia relativa `class_freq` y del peso crudo `raw_w = 1/√(class_freq + ε)` — receta directa que ya está justificada arriba en el bloque de scaffolding). Lo que **tenés que hacer vos** es el último paso: normalizar `raw_w` para que sus valores sumen exactamente `NUM_CLASSES` (eso mantiene la pérdida en un orden de magnitud razonable y evita retunear el learning rate) y guardarlo como tensor PyTorch en una variable llamada **`weights`** (`dtype=torch.float32`).
3. **Modelo, loss y optimizador:**
   - Instanciá la U-Net con 3 canales de entrada y `NUM_CLASSES` de salida, y mandala al `device`.
   - Definí la pérdida como cross-entropy multiclase **ponderada** (`nn.CrossEntropyLoss`) con los pesos del paso anterior. Asegurate de pasarle también la opción para que ignore los píxeles marcados con `IGNORE_INDEX` — sin eso, los píxeles del borde contribuyen al gradiente como si fueran clase 255 (que ni siquiera existe) y rompen el entrenamiento.
   - Como optimizador usá Adam (`torch.optim.Adam`) con learning rate 1e-3.
4. **Loop de entrenamiento:** entrená por 4 epochs. En cada epoch:
   - Modo train. Para cada batch del iterador de train:
     - Mandá imagen y máscara al device. Convertí la máscara a `long` (la cross-entropy exige índices enteros como target).
     - Forward por la red. La salida tiene shape `(B, NC, 256, 256)` — exactamente la misma resolución espacial que la máscara, así que se pueden comparar directamente sin recortar nada.
     - Calculá la pérdida, backpropagá, hacé el step del optimizador y limpiá los gradientes.
     - Acumulá lo necesario para reportar pérdida promedio y accuracy de pixel al final del epoch.
   - Modo eval. Recorré el iterador de validación midiendo accuracy de pixel sobre val. No olvides envolver el bloque en un contexto `no_grad` para no acumular gradientes inútilmente.
   - Imprimí una línea por epoch con `epoch | train_loss | train_acc | val_acc`.
   - **Guardá el `val_acc` de cada epoch en una lista llamada `val_acc_history`** — la vamos a comparar con los resultados del Ej. 10 (fine-tuning).

> **Pista — accuracy de pixel:** comparar `prediccion == ground_truth` (después del `argmax` sobre canales) te da un tensor booleano. Sumá los `True` y dividí por la cantidad de píxeles "no-ignorables". Para excluir del conteo los píxeles `IGNORE_INDEX`, hacé un AND con una máscara booleana que marque los píxeles válidos del ground truth.
>
> **Nota — qué esperar:** con la receta del lab (subset de 1500 imágenes, 4 epochs, U-Net con padding=1 + BatchNorm, pesos `1/√freq`, augmentation por flip horizontal) `val_acc` debería arrancar en ~0.72-0.78 en epoch 1 y llegar a **~0.78-0.85** en epoch 4. La línea de base "predecir todo fondo" sería ~0.70, así que cualquier valor claramente por encima indica que la red discrimina mascota de fondo, aunque sea de forma rudimentaria. Los resultados verdaderamente buenos van a aparecer en el Ej. 10.

In [ ]:
# ─── 1) Conteo de píxeles por clase (PRE-RESUELTO) ──────────────────────────
# Recorremos pet_train acumulando cuántos píxeles tiene cada clase. Ignoramos
# los píxeles marcados como IGNORE_INDEX. Tarda 1-2 min la primera vez (toca
# abrir cada imagen del dataset).
freqs = Counter()
for _, mask in pet_train:
    valid = mask[mask != IGNORE_INDEX]
    unique, counts = torch.unique(valid, return_counts=True)
    for c, n in zip(unique.tolist(), counts.tolist()):
        freqs[c] += n
total_pixels = sum(freqs.values())
print(f"Píxeles totales (sin IGNORE_INDEX): {total_pixels:,}")

df = pd.DataFrame({
    "clase":   PET_CLASSES,
    "píxeles": [freqs.get(i, 0) for i in range(NUM_CLASSES)],
    "freq":    [round(freqs.get(i, 0) / total_pixels, 4)
                for i in range(NUM_CLASSES)],
})
print(df.to_string(index=False))

# ─── 2) Pesos: 1 / sqrt(freq) normalizado a sumar NUM_CLASSES ──────────────
# Las dos primeras líneas vienen pre-resueltas porque son receta directa.
# Lo que tenés que hacer vos es la NORMALIZACIÓN final + convertir a tensor.
class_freq = np.array([freqs.get(i, 0) / total_pixels
                       for i in range(NUM_CLASSES)])
raw_w = 1.0 / np.sqrt(class_freq + 1e-6)            # epsilon evita 0/0

# ── Escalá `raw_w` para que sus valores sumen exactamente NUM_CLASSES.
# Pista: si querés que un vector v sume a S, multiplicalo por S / v.sum().
# Convertí el resultado a torch.tensor float32 y guardalo en `weights`.
# Tu código aquí
# weights = ...

# Imprimimos los pesos resultantes para verificar (no toques esto).
print(f"\nPesos por clase (normalizados a sumar NUM_CLASSES={NUM_CLASSES}):")
for i, n in enumerate(PET_CLASSES):
    print(f"  {n:14s}  freq={class_freq[i]:.4f}  weight={weights[i].item():.3f}")

# ─── 3) Modelo, loss y optimizador ──────────────────────────────────────────
# - model: UNet(input_channel=3, num_classes=NUM_CLASSES) en device.
# - criterion: nn.CrossEntropyLoss(weight=..., ignore_index=IGNORE_INDEX).
#   Acordate de mover los pesos al device.
# - optimizer: torch.optim.Adam con lr=1e-3.

# Tu código aquí

# ─── 4) Loop de entrenamiento (4 epochs, guardando val_acc_history) ────────
# Estructura sugerida:
#   num_epochs = 4
#   val_acc_history = []
#   for epoch in range(num_epochs):
#       model.train()
#       L_sum, n_correct, n_valid = 0.0, 0, 0
#       for X, y in train_iter:
#           ... mover a device, y.long(), forward, loss, backward, step, zero_grad
#           ... actualizar L_sum, n_correct, n_valid usando la máscara (y != IGNORE_INDEX)
#       train_loss = L_sum / len(pet_train)
#       train_acc  = n_correct / n_valid
#
#       model.eval()
#       n_correct_v, n_valid_v = 0, 0
#       with torch.no_grad():
#           for X, y in val_iter:
#               ... forward, argmax, contar correctos sobre píxeles válidos
#       val_acc = n_correct_v / n_valid_v
#       val_acc_history.append(val_acc)
#       print(f"epoch {epoch+1}/{num_epochs}  train_loss=...  train_acc=...  val_acc=...")

# Tu código aquí

**Pregunta de análisis:**

Si en lugar de usar pesos por clase entrenaras la misma red con `nn.CrossEntropyLoss()` "plana" (sin `weight`), ¿qué esperarías que pase con la **accuracy de pixel global** y con la **accuracy de pixel sobre la mascota**? Justificá.

*(Escribí tu respuesta acá)*

---
## Sección E: Predicción y visualización

Ya tenemos un modelo entrenado. Vamos a usarlo para producir máscaras predichas sobre imágenes del split de validación y compararlas con el ground truth.

### Ejercicio 9 — Visualizar predicciones

**Objetivo:** Tomar imágenes del split de validación, predecir sus máscaras con la red entrenada y mostrar la triple "imagen original / predicción / ground truth" para inspeccionar visualmente qué tan bien funciona el modelo.

**Enunciado:**

La celda viene con dos ayudas pre-resueltas para que te concentres en la parte que importa:

- `trimap_to_index` — el mapeo trimap `{1, 2, 3}` → índices `{1, 0, 255}` (es exactamente el mismo que aplica `PetSegDataset.__getitem__`; reimplementarlo no aporta nada).
- El armado de las dos figuras y la llamada a `add_seg_legend` en cada una — boilerplate de matplotlib.

Lo que tenés que completar:

1. **`label2image`** — función que recibe un tensor 2D `(H, W)` con índices de clase y devuelve un tensor 3D `(H, W, 3)` con los colores RGB correspondientes según `PET_COLORMAP`. Para los píxeles marcados como `IGNORE_INDEX`, podés pintar un color "neutro" (gris) — no es crítico porque son pocos.
2. **`predict_and_render`** — función que toma un índice y la fila de axes donde dibujar. Tu trabajo dentro:
   - Recortá un parche 256×256 **desde el centro** de la imagen y del trimap. Centrar el recorte (en lugar de tomarlo desde una esquina) maximiza la chance de que la mascota quede dentro del cuadro.
   - Normalizá la imagen con la misma media/std de ImageNet que usaba el dataset durante el entrenamiento (la celda ya define un objeto `norm` con esos valores).
   - Pasala por la red en modo eval (dentro de un bloque `no_grad`) y quedate con la clase de mayor logit por píxel (`argmax` sobre canales).
   - Mapeá el ground truth con `trimap_to_index` (ya provisto) antes de visualizar — el trimap viene con valores `{1, 2, 3}` y nuestro modelo predice `{0, 1}`.
   - Dibujá en las tres axes recibidas: imagen, predicción y ground truth (usando `label2image` para los dos últimos).

> **Pista:** En `torchvision.transforms.functional` hay una función `crop` que toma una imagen y devuelve un crop a partir de coordenadas `top`, `left`, `height` y `width`. Para centrar el crop usá `top = (H - 256) // 2` y `left = (W - 256) // 2`.
>
> **Nota:** la celda ya incluye un mini "resize de seguridad" para imágenes más chicas que 256×256 (raro en Pet, pero hay un puñado). Eso te lo dejamos hecho.

In [ ]:
# ─── 1) label2image: índices de clase → imagen RGB ─────────────────────────
def label2image(pred):
    """
    Mapea un tensor 2D (H, W) de índices de clase a una imagen RGB (H, W, 3)
    usando PET_COLORMAP. Los píxeles IGNORE_INDEX se pintan de gris.

    Idea clave: en PyTorch podés indexar un tensor con otro tensor de
    índices ("fancy indexing"). Si tenés `colormap` de forma (NC, 3) y
    `pred` de forma (H, W) con valores 0..NC-1, entonces
    `colormap[pred.long()]` te devuelve un tensor (H, W, 3) con cada
    píxel pintado del color RGB de su clase.
    """
    # Convertimos PET_COLORMAP a tensor (NUM_CLASSES, 3) uint8 (PRE-RESUELTO).
    colormap = torch.tensor(PET_COLORMAP, device=pred.device, dtype=torch.uint8)

    # Los píxeles IGNORE_INDEX (=255) están fuera del rango 0..NC-1 y
    # romperían el indexing. Los pasamos a 0 en una COPIA — después
    # parcheamos esos píxeles a gris en el resultado (PRE-RESUELTO).
    safe = pred.clone()
    safe[safe == IGNORE_INDEX] = 0

    # ── Indexá `colormap` con `safe.long()` para obtener `img` (H, W, 3).
    # Tu código aquí
    # img = ...

    # ── Pintá de gris (128, 128, 128) los píxeles donde `pred == IGNORE_INDEX`.
    # Pista: podés asignar un tensor de tres valores RGB usando indexación
    # booleana sobre `img`.
    # Tu código aquí

    return img.cpu()


# ─── trimap_to_index: trimap {1,2,3} → índices {1, 0, 255}  (PRE-RESUELTO) ─
def trimap_to_index(trimap):
    """Mismo mapeo que aplica PetSegDataset.__getitem__."""
    out = torch.full_like(trimap, IGNORE_INDEX)
    out[trimap == 2] = 0
    out[trimap == 1] = 1
    return out


# ─── 2) Lectura de imágenes + transformación de normalización ──────────────
n = 4
test_imgs, test_masks = read_pet_images(pet_dir, n, split='test')
model.eval()

# Misma media/std que aplicaba PetSegDataset durante el train.
norm = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225])


def predict_and_render(idx, ax_row):
    """Procesa la imagen idx-ésima y la dibuja en la fila de axes dada."""
    img_t = test_imgs[idx]
    msk_t = test_masks[idx].squeeze(0)            # (1, H, W) → (H, W)
    H, W = img_t.shape[1], img_t.shape[2]

    # Resize de seguridad si la imagen es más chica que 256 (PRE-RESUELTO).
    if H < 256 or W < 256:
        f = max(256 / H, 256 / W) * 1.1
        img_t = transforms.functional.resize(
            img_t, [int(H * f), int(W * f)], antialias=True)
        msk_t = transforms.functional.resize(
            msk_t.unsqueeze(0), [int(H * f), int(W * f)],
            interpolation=transforms.InterpolationMode.NEAREST).squeeze(0)
        H, W = img_t.shape[1], img_t.shape[2]

    # ── Centro-crop 256x256 sobre img_t y msk_t.
    # Tu código aquí
    # img_crop  = ...
    # mask_crop = ...

    # ── Normalizá img_crop con `norm`, agregá dimensión de batch, mandá al
    # device. Pasalo por la red en modo eval (no_grad) y tomá el argmax
    # sobre canales para obtener `pred` (256, 256).
    # Tu código aquí
    # X    = ...
    # pred = ...

    # ── Ground truth: usá trimap_to_index(mask_crop.long()).
    # Tu código aquí
    # gt_idx = ...

    # ── Dibujo en las 3 axes: imagen / predicción / ground truth.
    # Recordá que matplotlib espera (H, W, C). Para la imagen usá
    # img_crop.permute(1, 2, 0). Para pred y gt_idx usá label2image(...).
    # Tu código aquí


# ─── 3) Dos figuras de 2 filas cada una (PRE-RESUELTO) ─────────────────────
ROWS_PER_FIG = 2
with torch.no_grad():
    for fig_idx in range(0, n, ROWS_PER_FIG):
        fig, axs = plt.subplots(ROWS_PER_FIG, 3, figsize=(12, 4 * ROWS_PER_FIG))
        for r in range(ROWS_PER_FIG):
            predict_and_render(fig_idx + r, axs[r])
        add_seg_legend(fig)
        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.show()

**Pregunta de análisis:**

Comparando las predicciones con el ground truth: ¿en qué regiones de la imagen la red funciona mejor y en cuáles peor? Proponé al menos dos mejoras concretas del pipeline (datos, arquitectura o entrenamiento) que apunten a los puntos débiles que detectes.

*(Escribí tu respuesta acá)*

---
## Sección F: Fine-tuning desde una U-Net pre-entrenada

Hasta acá entrenamos la U-Net **desde cero**. La prueba inicial del Ej. 8 muestra que la red arranca — supera el baseline trivial de "todo fondo" — pero los resultados son modestos y las predicciones del Ej. 9 lo confirman: blobs aproximados, contornos imprecisos, mascotas chicas que se diluyen. La razón es bien conocida: una red profunda con ~31M de parámetros no puede aprender features visuales útiles a partir de 1500 imágenes. Necesitaría órdenes de magnitud más datos, mucho más cómputo, augmentation pesada o todo lo anterior.

La receta práctica de la última década resuelve esto con **transfer learning**: en lugar de inicializar los pesos del encoder al azar, se reemplazan por los pesos de una red pre-entrenada sobre ImageNet (1.2 millones de imágenes naturales etiquetadas con 1000 clases). El encoder ya viene "sabiendo" extraer bordes, texturas y partes de objetos — y solo hay que afinar esos features para la tarea específica de segmentación. Las primeras epochs del fine-tuning ya producen resultados que el train desde cero no alcanza ni con muchísimas más epochs.

Para implementarlo vamos a usar la librería `segmentation_models_pytorch` (smp), que es el estándar de facto en PyTorch para arquitecturas de segmentación con encoders pre-entrenados. Su clase `smp.Unet` arma una U-Net cuyo encoder es una CNN clásica (ResNet, EfficientNet, etc.) cargada con pesos de ImageNet, y cuyo decoder replica la parte estándar de U-Net que ya implementaste a mano (con padding y BN).

> **Importante — la normalización ya está bien:** `PetSegDataset` aplica `transforms.Normalize` con la media/std de ImageNet, que es exactamente lo que esperan los encoders pre-entrenados de smp. **No hace falta tocar nada del dataset ni de los DataLoaders** — entran tal cual al modelo nuevo. Si hubiéramos normalizado con otras estadísticas, el transfer learning andaría peor porque las activaciones de las primeras capas verían distribuciones para las que el encoder no fue pre-entrenado.

In [ ]:
# ─── Limpieza antes del fine-tuning ─────────────────────────────────────────
# Vamos a tener dos modelos en GPU al mismo tiempo (el desde cero del Ej. 8
# y el fine-tuning del Ej. 10) porque después los comparamos en visualización.
# Forzamos GC y vaciamos la cache de CUDA antes de instanciar el segundo.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM libre antes del fine-tuning: "
          f"{torch.cuda.mem_get_info()[0] / 1e9:.2f} GB de "
          f"{torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

### Ejercicio 10 — Fine-tuning de una U-Net con encoder pre-entrenado

**Objetivo:** Repetir el entrenamiento de la Sección D pero partiendo de una U-Net cuyo encoder está pre-entrenado sobre ImageNet, comparar cuantitativa y visualmente con la red entrenada desde cero.

**Enunciado:**

La celda viene con las partes 5 y 6 (comparación cuantitativa y visual) **pre-resueltas** — son puro boilerplate de pandas y matplotlib que no agregan nada nuevo al pipeline. Vos te encargás de las partes 1 a 4, que es lo que tiene contenido conceptual:

1. **Instalación de `segmentation_models_pytorch`.** En Colab no viene preinstalado: hay que correr `!pip install -q segmentation-models-pytorch`. La instalación toma unos segundos. Después importá la librería como `smp`.

2. **Modelo: `smp.Unet` con encoder ResNet34 pre-entrenado.** Pasale `encoder_name="resnet34"`, `encoder_weights="imagenet"`, `in_channels=3` y `classes=NUM_CLASSES`. Mandalo al `device` y guardalo en una variable llamada **`model_ft`** (las partes pre-resueltas la usan con ese nombre). La estructura interna es la de una U-Net (encoder-decoder con skip connections) similar a la que implementaste — la diferencia clave es que el encoder es un ResNet-34 con pesos pre-entrenados, en lugar de una secuencia de `DownConvolution` con pesos al azar.

3. **Loss y optimizador.** Reutilizá los pesos por clase del Ej. 8 (no hay que recalcularlos: el dataset es el mismo). Mismas hiperparámetros: `nn.CrossEntropyLoss` ponderada con `ignore_index=IGNORE_INDEX`, `torch.optim.Adam` con `lr=1e-3`. Llamalos **`criterion_ft`** y **`optimizer_ft`**.

4. **Loop de entrenamiento por 5 epochs.** Mismo patrón que el Ej. 8 (train + eval por epoch, accuracy de pixel sobre píxeles válidos). Imprimí una línea por epoch y guardá el `val_acc` de cada una en `val_acc_ft_history`. Definí también `num_epochs_ft = 5` antes del loop — la parte 5 lo usa para iterar.

> **Pista — el modelo de smp y el tuyo reciben/devuelven exactamente lo mismo:** un tensor `(B, 3, 256, 256)` normalizado entra, un tensor `(B, NUM_CLASSES, 256, 256)` de logits sale. Eso significa que el loop de entrenamiento del Ej. 8 funciona casi tal cual, solo cambiando los nombres de las variables (`model → model_ft`, `criterion → criterion_ft`, `optimizer → optimizer_ft`).
>
> **Nota — qué esperar:** con 5 epochs de fine-tuning sobre Pet, `val_acc` debería superar **0.90 ya en la primera epoch** y converger en torno a **0.93-0.96** en la última. Esa es la magnitud del salto: un encoder ImageNet le ahorra a la red años de entrenamiento sobre features visuales generales y le permite enfocarse exclusivamente en la tarea de segmentación.

In [ ]:
# ─── 1) Instalación de segmentation_models_pytorch ──────────────────────────
# En Colab no viene preinstalado. Si ya lo tenés, este comando es no-op.
# Después importá la librería como `smp`.

# Tu código aquí

# ─── 2) Modelo: smp.Unet con encoder ResNet34 pre-entrenado ────────────────
# Guardalo en `model_ft`. Mandalo al device.

# Tu código aquí

# ─── 3) Loss y optimizador (reutilizamos los pesos por clase del Ej. 8) ────
# `criterion_ft` con weights y ignore_index, `optimizer_ft` con Adam lr=1e-3.

# Tu código aquí

# ─── 4) Loop de entrenamiento (5 epochs, guardando val_acc_ft_history) ─────
# Definí num_epochs_ft = 5 y val_acc_ft_history = [] antes del loop.
# Mismo patrón que el Ej. 8 cambiando model/criterion/optimizer por *_ft.

# Tu código aquí

# ─── 5) Comparación cuantitativa: tabla val_acc desde cero vs FT (PRE-RESUELTO)
# val_acc_history viene del Ej. 8 (4 valores). val_acc_ft_history acaba de
# generarse (5 valores). Mostramos las 5 epochs lado a lado, dejando "—"
# cuando el desde-cero ya terminó.
rows = []
for e in range(num_epochs_ft):
    a_zc = f"{val_acc_history[e]:.4f}" if e < len(val_acc_history) else "—"
    a_ft = f"{val_acc_ft_history[e]:.4f}"
    rows.append({"epoch": e + 1,
                 "val_acc desde cero":  a_zc,
                 "val_acc fine-tuning": a_ft})
print("\nComparación val_acc por epoch:")
print(pd.DataFrame(rows).to_string(index=False))

# ─── 6) Comparación visual: imagen / desde cero / fine-tuning / GT (PRE-RESUELTO)
# Reusamos test_imgs/test_masks que ya leímos en el Ej. 9 (los volvemos a
# leer por las dudas), label2image, trimap_to_index y la lógica de centro-crop
# + normalización. La única diferencia con el Ej. 9 es que ahora dibujamos
# 4 columnas (agregamos la del fine-tuning) y corremos los dos modelos.
n = 4
test_imgs, test_masks = read_pet_images(pet_dir, n, split='test')
model.eval()
model_ft.eval()


def predict_compare(idx, ax_row):
    """
    Para la imagen idx-ésima del split de test, dibuja en ax_row (4 axes):
    imagen, predicción desde cero, predicción fine-tuning, ground truth.
    """
    img_t = test_imgs[idx]
    msk_t = test_masks[idx].squeeze(0)            # (1, H, W) → (H, W)
    H, W = img_t.shape[1], img_t.shape[2]
    if H < 256 or W < 256:
        f = max(256 / H, 256 / W) * 1.1
        img_t = transforms.functional.resize(
            img_t, [int(H * f), int(W * f)], antialias=True)
        msk_t = transforms.functional.resize(
            msk_t.unsqueeze(0), [int(H * f), int(W * f)],
            interpolation=transforms.InterpolationMode.NEAREST).squeeze(0)
        H, W = img_t.shape[1], img_t.shape[2]

    top  = (H - 256) // 2
    left = (W - 256) // 2
    img_crop  = transforms.functional.crop(img_t, top, left, 256, 256)
    mask_crop = transforms.functional.crop(msk_t, top, left, 256, 256).long()
    X = norm(img_crop.float() / 255).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_zc = model(X).argmax(dim=1).squeeze(0)        # desde cero
        pred_ft = model_ft(X).argmax(dim=1).squeeze(0)     # fine-tuning
    gt_idx = trimap_to_index(mask_crop)

    ax_row[0].imshow(img_crop.permute(1, 2, 0))
    ax_row[0].set_title("Imagen"); ax_row[0].axis('off')
    ax_row[1].imshow(label2image(pred_zc))
    ax_row[1].set_title("Pred. desde cero"); ax_row[1].axis('off')
    ax_row[2].imshow(label2image(pred_ft))
    ax_row[2].set_title("Pred. fine-tuning"); ax_row[2].axis('off')
    ax_row[3].imshow(label2image(gt_idx))
    ax_row[3].set_title("Ground truth"); ax_row[3].axis('off')


ROWS_PER_FIG = 2
with torch.no_grad():
    for fig_idx in range(0, n, ROWS_PER_FIG):
        fig, axs = plt.subplots(ROWS_PER_FIG, 4,
                                figsize=(16, 4 * ROWS_PER_FIG))
        for r in range(ROWS_PER_FIG):
            predict_compare(fig_idx + r, axs[r])
        add_seg_legend(fig)
        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.show()

**Pregunta de análisis:**

¿Por qué el fine-tuning con un encoder pre-entrenado en ImageNet (un dataset de **clasificación** de 1000 clases como "perro labrador", "gato siamés", "auto", "edificio") transfiere tan bien a una tarea aparentemente distinta como **segmentación binaria de mascotas**? Pensá específicamente en qué está aprendiendo el encoder durante el pre-entrenamiento y por qué esos features son útiles también para decidir, píxel a píxel, si pertenece a una mascota o al fondo.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] Los tests de `SimpleConvolution`, `DownConvolution`, `UpConvolution`, `LastConvolution`, `crop_img` y `UNet` pasan sin errores.
- [ ] Los entrenamientos del Ej. 8 (prueba inicial, 4 epochs) y del Ej. 10 (fine-tuning, 5 epochs) corrieron sin OOM. Si tuve OOM, reinicié el entorno y volví a ejecutar.
- [ ] La prueba inicial del Ej. 8 supera claramente el ~70% de "predecir todo fondo" (esperable: ~0.78-0.85). El fine-tuning del Ej. 10 lo supera con holgura (esperable: >0.90).
- [ ] La grilla de visualización del Ej. 9 muestra al menos una imagen donde la silueta de la mascota es reconocible, aunque sea de forma rudimentaria.
- [ ] La comparación visual del Ej. 10 muestra una mejora clara de la columna fine-tuning respecto a la columna desde cero.
- [ ] Respondí las preguntas de análisis (Ej. 1, 7, 8, 9, 10).
- [ ] No modifiqué ninguna celda fuera de las de actividad (ni las de test ni las de setup).

---
## ¡Listo!

Implementaste tu primera red de segmentación semántica de punta a punta. Practicaste:

- **Manejo del dataset Oxford-IIIT Pet** — formato trimap, mapeo a índices de clase, random crop consistente entre imagen y máscara, horizontal flip como augmentation, `ignore_index` para los bordes ruidosos.
- **Arquitectura U-Net moderna** — bloques de doble conv con `padding=1` y `BatchNorm`, downsampling con maxpool, upsampling con convolución transpuesta, skip connections que concatenan directamente (sin recortes, gracias al padding). Vimos también, a través del ejercicio de `crop_img`, qué hace la variante del paper original (sin padding) para alinear los skips.
- **Entrenamiento balanceado** — pesos por clase calculados como `1/√freq` (no `1/freq`, que produce pesos demasiado agresivos) para evitar el colapso a "todo fondo".
- **Predicción y visualización** — `label2image` para mapear índices a colores, comparación lado a lado con el ground truth.
- **Fine-tuning desde un encoder pre-entrenado** — usaste `segmentation_models_pytorch` con encoder ResNet34 inicializado con pesos de ImageNet. La diferencia respecto a entrenar desde cero quedó evidente: pocas epochs, mejor performance, contornos más finos.

Lo que hace que la U-Net siga vigente más de diez años después de su publicación es justamente lo que viste acá: una arquitectura **simple**, **simétrica** y **sin componentes exóticos**, que captura bien la receta general de las redes encoder-decoder con skip connections. Hoy se la sigue usando como punto de partida en imagen biomédica, satelital, microscopía y cualquier dominio donde la señal de entrenamiento sea escasa — casi siempre con un encoder pre-entrenado, como vimos en el ejercicio final.

Con esto cierra el bloque de **detección y segmentación** de la materia. Lo siguiente vamos a verlo en redes recurrentes y arquitecturas para datos secuenciales.